In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [2]:
import pandas as pd
import numpy as np
import random
import torch
import re
import nltk
from nltk.corpus import stopwords

import shap


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report


In [3]:
# Make stop words.
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))
stop_words.extend(['mrs', 'ms', 'mr', 'am', 'pm'])

[nltk_data] Downloading package stopwords to /home/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}

# Load the dataset
dataset = pd.read_csv("./dataSyntheticAll.csv")
dataset["label"] = dataset["needs"].map(label_map)

In [6]:
# Make function to remove punctuation, make lowercase, remove names, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        # lowercase + extract words only
        tokens = re.findall(r"\b[a-zA-Z]+\b", note.lower())
        
        # filter stopwords
        tokens = [t for t in tokens if t not in stop_words]
        
        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

In [7]:
dataset['report'] = preprocessing(dataset['report'].values.tolist())

In [8]:
# Split the dataset. (80/10/10)
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=dataset["label"]
)

# Check distributions.
def check_distribution(dataframe, name):
    counts = dataframe["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_df, "Train")
check_distribution(test_df, "Test")


Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Test distribution:
label
0    0.502161
1    0.497839
Name: proportion, dtype: float64


In [9]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X = vectorizer.fit_transform(train_df['report'].values.tolist())
y = train_df['label'].values.tolist()

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

preds = model.predict(vectorizer.transform(test_df['report'].values.tolist()))
labels = np.array(test_df['label'].values.tolist())

# Confusion Matrix
cm = confusion_matrix(labels, preds)
print("--------------- Confusion Matrix ---------------")
print(cm)

# Classification Report
report = classification_report(labels, preds)
print("--------------- Classification Report ---------------")
print(report)

--------------- Confusion Matrix ---------------
[[514  67]
 [ 28 548]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.95      0.88      0.92       581
           1       0.89      0.95      0.92       576

    accuracy                           0.92      1157
   macro avg       0.92      0.92      0.92      1157
weighted avg       0.92      0.92      0.92      1157



In [10]:
explainer = shap.Explainer(model, X)
shap_values = explainer(X)

In [11]:
mean_importance = np.abs(shap_values.values).mean(axis=0)

feature_names = vectorizer.get_feature_names_out()

top_features = sorted(
    zip(feature_names, mean_importance),
    key=lambda x: x[1],
    reverse=True
)[:100]

In [12]:
for word, contribution in top_features:
    print(word, float(contribution))

signs 0.10148486468491415
today 0.07564914994270677
family 0.054375016973804396
staff 0.05390316860197134
enjoyed 0.04121202328739294
increased 0.039659482439260305
afternoon 0.0395896409734192
exhibited 0.039065459458954496
visit 0.036234669900345226
review 0.03421135803123963
experiencing 0.03385017876152924
need 0.03381423459773764
palliative 0.03318249295110467
skin 0.03270575302078274
help 0.032652277668665325
exhibited signs 0.03262003319286236
frequent 0.032335442710618466
session 0.03225978375244998
morning 0.03212074024045307
palliative care 0.03179577882049631
resident 0.03078105959778041
severe 0.029340004801474175
care 0.029304300277724675
day 0.02819779187110297
given 0.026592573852043572
management 0.02588189306165664
requiring 0.02586451033524233
agitation 0.025784366703300567
participated 0.025566948867404057
persistent 0.025054698513391187
care plan 0.024951834028217743
complained 0.024735428489775783
needs 0.024717958914926976
support 0.02455423571874849
pain 0.024450